In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import json
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Funciona com kernel iniciado na raiz do projeto ou dentro de /notebooks.
# raiz_candidatas = [Path.cwd(), *Path.cwd().parents]
# ROOT = next(
#     (p for p in raiz_candidatas if (p / "models/train_sarimax.py").exists()),
#     None,
# )
# if ROOT is None:
#     raise FileNotFoundError(
#         "Não encontrei a raiz do PharmaDemand. Abra a pasta do projeto no VS Code "
#         "e selecione o kernel do ambiente virtual (.venv)."
#     )

# Facilita o import de src.models a partir da pasta notebooks/.
# if str(ROOT) not in sys.path:
#     sys.path.insert(0, str(ROOT))

# from models.train_sarimaxZ import (
#     train_and_evaluate,
#     SARIMAXRegressor
# )

PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

from src.config.config import MODELS_BASELINE,MODELS_PARAM,DATA_FEATURES,DATA_EXPORTS,DATA_PROCESSED
from src.repository.parquet_repository import ParquetRepository
from src.models.Sarimax import SARIMAXRegressor

parket_data = ParquetRepository(DATA_FEATURES)

regressor_model = SARIMAXRegressor (category="N02BE", horizon_days=7, maxiter=7)

CATEGORIA = "N02BE"
MESES_TESTE = 6
# CSV_PATH = ROOT / "data" / "raw" / "salesdaily.csv"
# OUT = ROOT / "outputs" / "sarimax" / CATEGORIA

# print("Projeto:", ROOT)
# print("Dataset:", CSV_PATH)
print("Categoria:", CATEGORIA)

Categoria: N02BE


In [2]:
# Parquet já carregado no DataFrame
df_load = parket_data.load("anomaly_predictions.parquet")
horizon_days = 7

df_train = df_load[(df_load["split"]=="train")]

[OK] Arquivo carregado: D:\PROJETOS\git_repo\PharmaDemand-ml\data\features\anomaly_predictions.parquet


In [3]:
df_train.head(10)

,date,category,horizon_days,forecast_origin,split,quantity,year,month,quarter,weekday,...,qty_roll_std_56,qty_mean_ratio_7_28,qty_mean_delta_7_28,qty_last_ratio_28,qty_last_zscore_28,qty_cv_28,qty_zero_ratio_28,qty_mean_ratio_14_56,anomaly_score,is_anomaly
0,2014-03-06,N02BE,7,2014-02-27,train,44.1,2014,3,1,3,...,14.292534,0.809034,-6.764286,0.937286,-0.174629,0.359129,0.0,1.047649,0.087633,0
1,2014-03-07,N02BE,7,2014-02-28,train,25.0,2014,3,1,4,...,14.094213,0.847190,-5.467857,1.034035,0.096526,0.352603,0.0,1.043780,0.080584,0
2,2014-03-08,N02BE,7,2014-03-01,train,27.6,2014,3,1,5,...,13.746840,0.938590,-2.214286,1.406101,1.139263,0.356460,0.0,1.080984,0.112373,0
3,2014-03-09,N02BE,7,2014-03-02,train,11.3,2014,3,1,6,...,14.089044,0.887603,-4.025000,0.184302,-2.189715,0.372513,0.0,1.039697,0.274512,0
4,2014-03-10,N02BE,7,2014-03-03,train,47.0,2014,3,1,0,...,14.084290,0.879081,-4.285714,0.623539,-0.983335,0.382841,0.0,1.028270,0.095044,0
5,2014-03-11,N02BE,7,2014-03-04,train,30.4,2014,3,1,1,...,13.592314,0.915195,-3.056250,1.311795,0.819677,0.380388,0.0,1.006306,0.106161,0
6,2014-03-12,N02BE,7,2014-03-05,train,19.6,2014,3,1,2,...,13.604102,0.886016,-4.077679,0.698829,-0.778093,0.387062,0.0,0.914075,0.042547,0
7,2014-03-13,N02BE,7,2014-03-06,train,15.5,2014,3,1,3,...,13.661151,0.927044,-2.616964,1.229421,0.592262,0.387364,0.0,0.954226,0.062473,0
8,2014-03-14,N02BE,7,2014-03-07,train,16.0,2014,3,1,4,...,13.401121,0.876113,-4.459821,0.694462,-0.798389,0.382694,0.0,0.968432,0.047920,0
9,2014-03-15,N02BE,7,2014-03-08,train,23.3,2014,3,1,5,...,13.127264,0.791908,-7.420536,0.773980,-0.581430,0.388730,0.0,0.985624,0.093407,0


In [ ]:
modelH = SARIMAXRegressor(
    category="N02BE",
    horizon_days=15,
    maxiter=400
)

model, best_params = modelH.tune_hyperparameters(df_train)

print(best_params)
print(model.cv_results_)
print(model.cv_failures_)
print("Convergência CV:", model.best_cv_converged)
print("Convergência final:", model.converged)

In [ ]:
model.save_best_params()

In [9]:
local = f"{MODELS_PARAM}\\sarimax_best_params.json"

# Carregar e aplicar os parâmetros
regressor_model.load_params()

df_train = df_load[(df_load["horizon_days"]== horizon_days )]

df_train.head(50)

,date,category,horizon_days,forecast_origin,split,quantity,year,month,quarter,weekday,...,qty_roll_std_56,qty_mean_ratio_7_28,qty_mean_delta_7_28,qty_last_ratio_28,qty_last_zscore_28,qty_cv_28,qty_zero_ratio_28,qty_mean_ratio_14_56,anomaly_score,is_anomaly
0,2014-03-06,N02BE,7,2014-02-27,train,44.10,2014,3,1,3,...,14.292534,0.809034,-6.764286,0.937286,-0.174629,0.359129,0.0,1.047649,0.087633,0
1,2014-03-07,N02BE,7,2014-02-28,train,25.00,2014,3,1,4,...,14.094213,0.847190,-5.467857,1.034035,0.096526,0.352603,0.0,1.043780,0.080584,0
2,2014-03-08,N02BE,7,2014-03-01,train,27.60,2014,3,1,5,...,13.746840,0.938590,-2.214286,1.406101,1.139263,0.356460,0.0,1.080984,0.112373,0
3,2014-03-09,N02BE,7,2014-03-02,train,11.30,2014,3,1,6,...,14.089044,0.887603,-4.025000,0.184302,-2.189715,0.372513,0.0,1.039697,0.274512,0
4,2014-03-10,N02BE,7,2014-03-03,train,47.00,2014,3,1,0,...,14.084290,0.879081,-4.285714,0.623539,-0.983335,0.382841,0.0,1.028270,0.095044,0
5,2014-03-11,N02BE,7,2014-03-04,train,30.40,2014,3,1,1,...,13.592314,0.915195,-3.056250,1.311795,0.819677,0.380388,0.0,1.006306,0.106161,0
6,2014-03-12,N02BE,7,2014-03-05,train,19.60,2014,3,1,2,...,13.604102,0.886016,-4.077679,0.698829,-0.778093,0.387062,0.0,0.914075,0.042547,0
7,2014-03-13,N02BE,7,2014-03-06,train,15.50,2014,3,1,3,...,13.661151,0.927044,-2.616964,1.229421,0.592262,0.387364,0.0,0.954226,0.062473,0
8,2014-03-14,N02BE,7,2014-03-07,train,16.00,2014,3,1,4,...,13.401121,0.876113,-4.459821,0.694462,-0.798389,0.382694,0.0,0.968432,0.047920,0
9,2014-03-15,N02BE,7,2014-03-08,train,23.30,2014,3,1,5,...,13.127264,0.791908,-7.420536,0.773980,-0.581430,0.388730,0.0,0.985624,0.093407,0


In [11]:
# Treinar com os parâmetros recuperados
modelC = regressor_model.fit_model(df_train)

In [ ]:
modelC,eva(df_train)

In [12]:
model, metrics, predictions = regressor_model.train_and_evaluate(
    df_load,
    category="N02BE",
    horizon_days=horizon_days,
    target_col="quantity",
    maxiter=200
)

# Tudo permanece em memória
df_predictions = predictions["exact_horizon"]
df_backtest = predictions["block_backtest"]

print(metrics["exact_horizon"]["sarimax"])

# Salvar somente o modelo
# model.save(
#     MODELS_BASELINE / "sarimax" / "N02BE" / "h15"
# )

{'n_predictions': 177, 'mae': 7.299729807253136, 'rmse': 9.899551165168955, 'wape_pct': 31.661993343011368, 'smape_pct': 32.84924823145195, 'r2': 0.2035755134721483}
